In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
from scipy import signal
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
import scipy.io as sio
from collections import Counter

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from xgboost import XGBClassifier

2026-07-13 16:12:41.408028: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783959161.698501      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783959161.783170      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783959162.493614      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783959162.493666      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783959162.493669      57 computation_placer.cc:177] computation placer alr

In [ ]:
import numpy as np, matplotlib.pyplot as plt
mat = sio.loadmat('/kaggle/input/datasets/sealeopard/unibo-semg/user_1_day_1_posture_1.mat')
emg = np.asarray(mat['emg'], dtype=np.float64)   # (samples, 4)
relabel = mat['relabel'].squeeze()

# 1) Nhìn một đoạn ngắn (0.5s = 250 mẫu) của 1 kênh — hình dạng nói lên tất cả
seg = emg[5000:5250, 0]
plt.figure(figsize=(12,3)); plt.plot(seg, marker='.', ms=3); plt.title('0.5s kênh 0'); plt.grid(alpha=.3); plt.show()

# 2) Đặc trưng phân biệt "mượt" vs "thô":
x = emg[:, 0]
# (a) toàn dương? (rectify) 
print("toàn dương:", (x >= 0).all(), "| min:", x.min())
# (b) độ "gai" — tỷ lệ mẫu mà tín hiệu đổi hướng (lên<->xuống). Thô ~0.5, mượt <<0.5
dx = np.diff(x)
turns = np.sum(np.diff(np.sign(dx)) != 0) / len(dx)
print(f"tỷ lệ đổi hướng: {turns:.3f}  (thô ~0.4-0.5, mượt << 0.2)")
# (c) tự tương quan trễ-1: mượt thì rất gần 1
ac1 = np.corrcoef(x[:-1], x[1:])[0,1]
print(f"autocorr lag-1: {ac1:.4f}  (mượt > 0.95, thô thấp)")
# (d) bao nhiêu mẫu để tín hiệu 'đổi đáng kể' — mượt thì lâu
print(f"std toàn bộ: {x.std():.1f} | std của diff: {np.diff(x).std():.1f} "
      f"| tỷ lệ diff/tín hiệu: {np.diff(x).std()/x.std():.4f}  (mượt << 1)")

In [ ]:
BASE_PATH   = '/kaggle/input/datasets/sealeopard/unibo-semg'
N_FOLDS     = 5
RANDOM_SEED = 200
FEATURES    = 'combine'

PARTICIPANT = 3
DAYS = [1, 2, 3, 4, 5, 6, 7, 8]
POSTURES = [1, 2, 3, 4]
MIN_TRIAL_LEN = 75

sample_rate = 500
num_channels = 4
zc_threshold   = 0.0
ssc_threshold  = 0.0

In [ ]:
def load_file_data_full(participant, day, posture, base_path=BASE_PATH):
    file = f'user_{participant}_day_{day}_posture_{posture}.mat'
    mat = sio.loadmat(os.path.join(base_path, file))
    emg     = mat['emg']
    label   = mat['label'].squeeze()
    relabel = mat['relabel'].squeeze()
    gc      = mat['gestureCounter'].squeeze()
    return emg, label, relabel, gc

def segment_stream(emg_cf, relabel, posture):
    diff = np.diff(relabel)
    starts = np.insert(np.where(diff != 0)[0] + 1, 0, 0)
    ends = np.append(starts[1:], len(relabel))
    trials = []
    for s, e in zip(starts, ends):
        if e - s < MIN_TRIAL_LEN:
            continue
        trials.append({'emg': emg_cf[:, s:e], 'gesture': int(relabel[s]), 'posture': int(posture)})
    return trials

def load_all_trials(subject, days, postures, base_path=BASE_PATH):
    X_list, y_gesture, y_posture = [], [], []
    for day in days:
        for posture in postures:
            emg, _label, relabel, _gc = load_file_data_full(subject, day, posture, base_path)
            emg_cf = emg.T                              # channels x samples
            for t in segment_stream(emg_cf, relabel, posture):
                X_list.append(t['emg'])
                y_gesture.append(t['gesture'])
                y_posture.append(t['posture'])
    y_gesture = np.array(y_gesture)
    y_posture = np.array(y_posture)
    print(f'Loaded {len(X_list)} trials | subject={subject} days={days} ')
    print('  trial count per gesture (all postures):',
          dict(sorted(Counter(y_gesture.tolist()).items())))
    return X_list, y_gesture, y_posture

def signal_normalize_window(window, eps=1e-8):
    return window / np.sqrt(np.mean(window ** 2) + eps)

def extract_hudgins_9_features(window):
    if window.ndim != 2:
        raise ValueError("Window must be a 2D array (channels × samples)")
        
    mean_val = np.mean(np.abs(window), axis=1) 
    zc = np.sum(
        (np.diff(np.sign(window), axis=1) != 0)
        & (np.abs(np.diff(window, axis=1)) >= zc_threshold),
        axis=1,
    )
    
    mid   = window[:, 1:-1]
    left  = window[:, :-2]
    right = window[:, 2:]
    ssc = np.sum(((mid - left) * (mid - right)) > ssc_threshold, axis=1)
    
    wl = np.sum(np.abs(np.diff(window, axis=1)), axis=1)

    rms = np.sqrt(np.mean(window**2, axis=1))
    var = np.var(window, axis=1)
    std = np.std(window, axis=1)
    mean_w = np.mean(window, axis=1, keepdims=True)
    std_w = np.where(std < 1e-8, 1e-8, std)[:, None]
    kurt = np.nan_to_num(np.mean(((window - mean_w) / std_w) ** 4, axis=1) - 3.0, nan=0.0)   
    skew = np.nan_to_num(np.mean(((window - mean_w) / std_w) ** 3, axis=1), nan=0.0)

    features = np.concatenate([mean_val, zc, ssc, wl, rms, var, std, kurt, skew])
    return features

def extract_hudgins_features(window):
    if window.ndim != 2:
        raise ValueError("Window must be a 2D array (channels × samples)")
        
    mean_val = np.mean(np.abs(window), axis=1) 
    zc = np.sum(
        (np.diff(np.sign(window), axis=1) != 0)
        & (np.abs(np.diff(window, axis=1)) >= zc_threshold),
        axis=1,
    )
    
    mid   = window[:, 1:-1]
    left  = window[:, :-2]
    right = window[:, 2:]
    ssc = np.sum(((mid - left) * (mid - right)) > ssc_threshold, axis=1)

    wl = np.sum(np.abs(np.diff(window, axis=1)), axis=1)

    features = np.concatenate([mean_val, zc, ssc, wl])
    return features

def extract_sntdf_features(window, eps=1e-8):
    
    d1 = np.diff(window, axis=1)
    d2 = np.diff(window, n=2, axis=1)
    d3 = np.diff(window, n=3, axis=1) 

    mean_val = np.log(np.mean(np.abs(window), axis=1) + eps)
    p0 = np.log(np.sum(window**2, axis=1) + eps)
    p2 = np.log(np.sum(d1**2, axis=1) + eps)
    p4 = np.log(np.sum(d2**2, axis=1) + eps)
    p6 = np.log(np.sum(d3**2, axis=1) + eps)

    ac1 = np.log(np.mean(np.abs(d1), axis=1) + eps)
    ac2 = np.log(np.mean(np.abs(d2), axis=1) + eps)

    corr = np.corrcoef(window)
    corr = np.nan_to_num(corr, nan=0.0) 
    corr_flat = corr[np.triu_indices(num_channels, k=1)] # Upper triangle

    features = np.concatenate([mean_val, p0, p2, p4, p6, ac1, ac2, corr_flat])

    return features
    
def extract_features(emg_seg, features='hudgin'):
    
    if features == 'hudgin':
        return extract_hudgins_features(emg_seg)
    elif features == 'hudgin_9':
        return extract_hudgins_9_features(emg_seg)
    elif features == 'sntdf':
        return extract_sntdf_features(signal_normalize_window(emg_seg))
    elif features == 'combine':
        return np.concatenate([
            extract_sntdf_features(signal_normalize_window(emg_seg)),
            extract_hudgins_9_features(emg_seg),
        ])
    else:
        raise ValueError(f"Unsupported features: {features}")

def build_trial_dataset(X_list, y_gesture, y_posture, trial_idx_list, features=FEATURES):
    feats_list, gesture_labels, posture_labels = [], [], []
    for ti in trial_idx_list:
        feat = extract_features(X_list[ti], features)   # 1-D, ONE per trial
        feats_list.append(feat)
        gesture_labels.append(y_gesture[ti])
        posture_labels.append(y_posture[ti])
    X = np.vstack(feats_list)
    return X, np.array(gesture_labels), np.array(posture_labels)

def generate_folds_per_posture(y_gesture, y_posture, n_folds=N_FOLDS, seed=RANDOM_SEED):
    y_gesture = np.asarray(y_gesture); y_posture = np.asarray(y_posture)
    all_idx = np.arange(len(y_gesture)); fold_results = {}
    for pos in np.unique(y_posture):
        m = (y_posture == pos); pos_idx = all_idx[m]; pos_g = y_gesture[m]
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        fold_results[pos] = {}
        for fold, (tr, te) in enumerate(skf.split(np.zeros(len(pos_g)), pos_g)):
            fold_results[pos][fold] = {'train_trial_idx': pos_idx[tr], 'test_trial_idx': pos_idx[te]}
    return fold_results
 
 
def merge_postures(fold_results, fold_num, postures_to_use=None):
    if postures_to_use is None:
        postures_to_use = sorted(fold_results.keys())
    train_idx, test_idx = [], []
    for pos in postures_to_use:
        train_idx.extend(fold_results[pos][fold_num]['train_trial_idx'].tolist())
        test_idx.extend(fold_results[pos][fold_num]['test_trial_idx'].tolist())
    return np.array(train_idx), np.array(test_idx)
 
 
def print_fold_balance_check(fold_results, y_gesture, y_posture, fold_num=0):
    y_gesture = np.asarray(y_gesture)
    print(f'\n=== Fold {fold_num} balance (per posture) ===')
    for pos in sorted(fold_results.keys()):
        tr = fold_results[pos][fold_num]['train_trial_idx']
        te = fold_results[pos][fold_num]['test_trial_idx']
        print(f'  posture {pos}: train={len(tr)} {dict(sorted(Counter(y_gesture[tr].tolist()).items()))} | '
              f'test={len(te)} {dict(sorted(Counter(y_gesture[te].tolist()).items()))}')

def normalize_data(X_train, X_test):
    scaler = StandardScaler()
    return scaler.fit_transform(X_train), scaler.transform(X_test)

class MLP48(BaseEstimator, ClassifierMixin):
    def __init__(self, input_dim=None, n_classes=None, lr=1e-3, epochs=30, batch_size=32, verbose=0):
        self.input_dim = input_dim
        self.n_classes = n_classes
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.verbose = verbose
        self.model = None
        self.loss_history = []

    def _build_model(self):
        model = Sequential()
        model.add(Dense(48, activation='relu', input_dim=self.input_dim))
        model.add(BatchNormalization())
        model.add(Dropout(0.3))
        model.add(Dense(24, activation='relu'))
        model.add(BatchNormalization())
        model.add(Dropout(0.3))
        model.add(Dense(self.n_classes, activation='softmax'))
        model.compile(optimizer=Adam(learning_rate=self.lr),
                      loss='categorical_crossentropy', metrics=['accuracy'])
        return model

    def fit(self, X, y):
        if self.input_dim is None:
            self.input_dim = X.shape[1]
        if self.n_classes is None:
            self.n_classes = len(np.unique(y))
        y_cat = to_categorical(y, num_classes=self.n_classes)
        self.model = self._build_model()
        history = self.model.fit(X, y_cat, epochs=self.epochs,
                                 batch_size=self.batch_size, verbose=self.verbose)
        self.loss_history = history.history['loss']
        return self

    def predict(self, X):
        return np.argmax(self.model.predict(X, verbose=0), axis=1)


def meta_learner_ensemble(X_train, y_train, X_test, random_state=42, cv=4):
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    n_cls = len(np.unique(y_train_enc))

    svc = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=random_state)
    xgb = XGBClassifier(
        eval_metric='mlogloss', n_estimators=100, max_depth=4,
        learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        random_state=random_state, verbosity=0
    )
    mlp48 = MLP48(input_dim=X_train.shape[1], n_classes=n_cls, epochs=30, verbose=0)
    base_models = [svc, xgb, mlp48]

    cv_split = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    oof_probas  = np.zeros((X_train.shape[0], n_cls * len(base_models)))
    test_probas = np.zeros((X_test.shape[0],  n_cls * len(base_models)))

    for i, model in enumerate(base_models):
        oof_fold_preds  = np.zeros((X_train.shape[0], n_cls))
        test_fold_preds = np.zeros((cv, X_test.shape[0], n_cls))

        for j, (tr_idx, val_idx) in enumerate(cv_split.split(X_train, y_train_enc)):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr        = y_train_enc[tr_idx]

            if isinstance(model, MLP48):
                tf.random.set_seed(random_state + j)
                m = MLP48(input_dim=X_train.shape[1], n_classes=n_cls, epochs=30, verbose=0)
                m.fit(X_tr, y_tr)
                val_pred  = m.model.predict(X_val,  verbose=0)
                test_pred = m.model.predict(X_test, verbose=0)
            else:
                m = model.__class__(**model.get_params())
                m.fit(X_tr, y_tr)
                val_pred  = m.predict_proba(X_val)
                test_pred = m.predict_proba(X_test)

            oof_fold_preds[val_idx] = val_pred
            test_fold_preds[j]      = test_pred

        oof_probas[:,  i*n_cls:(i+1)*n_cls] = oof_fold_preds
        test_probas[:, i*n_cls:(i+1)*n_cls] = test_fold_preds.mean(axis=0)

    scaler = MinMaxScaler()
    X_meta_train = scaler.fit_transform(oof_probas)
    X_meta_test  = scaler.transform(test_probas)

    meta_model = LogisticRegression(max_iter=500, random_state=random_state)
    meta_model.fit(X_meta_train, y_train_enc)

    y_pred_train = le.inverse_transform(meta_model.predict(X_meta_train))
    y_pred_test  = le.inverse_transform(meta_model.predict(X_meta_test))
    return y_pred_train, y_pred_test

def hmc_classification(X_train, X_test, yg_train, yg_test, yp_train, yp_test):
    X_train = np.asarray(X_train, dtype=np.float64)
    X_test  = np.asarray(X_test,  dtype=np.float64)
    X_train_norm, X_test_norm = normalize_data(X_train, X_test)

    # === Posture: ensemble meta-learner thay cho LDA/SVM ===
    pos_pred_train, pos_pred_test = meta_learner_ensemble(
        X_train_norm, yp_train, X_test_norm,
        random_state=RANDOM_SEED, cv=4
    )
    pos_acc = accuracy_score(yp_test, pos_pred_test)

    pos_pred_norm_test  = (pos_pred_test  - pos_pred_test.min()) / (pos_pred_test.max()  - pos_pred_test.min()  + 1e-8)
    pos_pred_norm_train = (pos_pred_train - pos_pred_train.min()) / (pos_pred_train.max() - pos_pred_train.min() + 1e-8)
    X_aug_train = np.hstack((X_train_norm, pos_pred_norm_train.reshape(-1, 1)))
    X_aug_test  = np.hstack((X_test_norm,  pos_pred_norm_test.reshape(-1, 1)))

    # === Gesture: vẫn dùng LDA theo từng posture ===
    grasp_clfs = {}
    for pos in np.unique(yp_train):
        mask = (yp_train == pos)
        if len(np.unique(yg_train[mask])) > 1:
            clf_g = LinearDiscriminantAnalysis()
            clf_g.fit(X_aug_train[mask], yg_train[mask])
            grasp_clfs[pos] = clf_g

    preds_grasp = []
    for i in range(len(X_aug_test)):
        p_pred = pos_pred_test[i]
        x_aug  = X_aug_test[i].reshape(1, -1)
        preds_grasp.append(grasp_clfs[p_pred].predict(x_aug)[0])

    preds_grasp = np.array(preds_grasp)
    multi_label_acc = np.mean((pos_pred_test == yp_test) & (preds_grasp == yg_test))
    soft_acc = accuracy_score(yg_test, preds_grasp)
    return pos_acc, multi_label_acc, soft_acc

In [ ]:
if __name__ == '__main__':
    print(f'=== UNIBO-INAIL HMC — subject {PARTICIPANT}, days {DAYS} ===\n')
 
    X_list, y_gesture, y_posture = load_all_trials(PARTICIPANT, DAYS, POSTURES, BASE_PATH)
 
    fold_results = generate_folds_per_posture(y_gesture, y_posture)
    print_fold_balance_check(fold_results, y_gesture, y_posture, fold_num=0)
 
    fold_metrics = []
    for fold in range(N_FOLDS):
        train_idx, test_idx = merge_postures(fold_results, fold)
        assert len(set(train_idx) & set(test_idx)) == 0, f'TRIAL LEAKAGE in fold {fold}'
 
        X_train, yg_train, yp_train = build_trial_dataset(X_list, y_gesture, y_posture, train_idx, FEATURES)
        X_test,  yg_test,  yp_test  = build_trial_dataset(X_list, y_gesture, y_posture, test_idx,  FEATURES)
 
        missing = set(np.unique(yp_test)) - set(np.unique(yp_train))
        if missing:
            print(f'  [!] postures {missing} in TEST but not TRAIN (fold {fold})')
 
        pos_acc, ml_acc, soft_acc = hmc_classification(
            X_train, X_test, yg_train, yg_test, yp_train, yp_test)
        print(f'Fold {fold}: train_trials={X_train.shape[0]} test_trials={X_test.shape[0]} '
              f'(dim={X_train.shape[1]}) | PostureAcc={pos_acc:.4f} MultiLabel={ml_acc:.4f} Soft={soft_acc:.4f}')
        fold_metrics.append((pos_acc, ml_acc, soft_acc))
 
    fold_metrics = np.array(fold_metrics)
    means = fold_metrics.mean(axis=0)
    print(f'\n{"="*60}\nMEAN over {N_FOLDS} folds (1 vector/trial)\n{"="*60}')
    print(f'  Posture Acc     : {means[0]:.4f}')
    print(f'  Multi-label Acc : {means[1]:.4f}')
    print(f'  Soft Acc        : {means[2]:.4f}')
    print('\n=== Done ===')